In [1]:
import sys
print(sys.executable)

c:\Users\usama\miniconda3\envs\vt_env\python.exe


In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
def find_root(marker="data"):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"no parent of {Path.cwd()} contains a '{marker}' folder")

ROOT = find_root()
WORKBOOK = ROOT / "data" / "raw" / "CO_HOT-Spring-Geothermometry-Template_2012-2-16.xlsx"
SHEET = "ThermalSpringGeothermometry"

print("root :", ROOT)
print("found :", WORKBOOK.exists())

root : c:\Dev\k-means-geothermal-identification\src
found : True


In [4]:
raw=pd.read_excel(WORKBOOK, sheet_name=SHEET, skiprows=[1])

In [5]:
raw.shape

(416, 87)

In [6]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 416 entries, 0 to 415
Data columns (total 87 columns):
 #   Column                                                                         Non-Null Count  Dtype  
---  ------                                                                         --------------  -----  
 0   ThermalSpringURI                                                               413 non-null    object 
 1   Name                                                                           412 non-null    str    
 2   Label                                                                          402 non-null    str    
 3   OtherName                                                                      402 non-null    str    
 4   OtherIdentifier                                                                402 non-null    object 
 5   Source                                                                         402 non-null    str    
 6   SourceURI                            

In [7]:
# Extracting related columns for the study
Column_map = {"Name": "Name", "FeatureType": "Type", "LatDegree": "Latitude", "LongDegree": "Longitude",
"Temperature": "Temperature", "Well Depth": "Depth", "Conductivity": "Cond", "pH": "pH", "TDS": "TDS", 
"Ca": "Ca", "Mg": "Mg", "Na": "Na", "K": "K", "SiO2": "SiO2", "Giggenbach Maturity Classification": "Equilibrium"}

In [8]:
# Checking for unique values of equilibrium maturity levels
raw['Giggenbach Maturity Classification'].unique()

<StringArray>
['IM', nan, 'FE', 'PE', 'PE/IM', 'PE ', 'FE/PE', 'FE ']
Length: 8, dtype: str

In [9]:
# Need to extract those maturity levels on fully and partial equilibrium
EQ_levels = {"IM", "FE", "PE", "PE/IM", "FE/PE"}

In [10]:
# Focusing on extracted column
raw = raw.loc[:, list(Column_map.keys())].rename(columns=Column_map)
raw.head()

,Name,Type,Latitude,Longitude,Temperature,Depth,Cond,pH,TDS,Ca,Mg,Na,K,SiO2,Equilibrium
0,Antelope Warm Spring,Thermal Spring,37.74333,-107.03722,32.0,NaN,180.0,NaN,151.0,4.0,0.3,44.0,0.1,41.0,IM
1,Antelope Warm Spring,Thermal Spring,37.74333,-107.03722,32.0,NaN,160.0,8.9,150.0,1.7,0.6,43.0,0.3,39.0,IM
2,Antelope Warm Spring,Thermal Spring,37.74333,-107.03722,32.0,NaN,234.0,9.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Axial,Well,40.30028,-107.78417,22.0,3.6,1750.0,7.1,1250.0,140.0,140.0,71.0,14.0,18.0,IM
4,Birdsie Warm Spring,Thermal Spring,37.72833,-107.05361,30.0,NaN,200.0,8.6,168.0,4.0,0.1,42.0,0.5,50.0,NaN


In [11]:
# Identifying EQ levels from the data of IM and (FE, PE)
raw["Equilibrium"] = raw["Equilibrium"].astype(str).str.strip()
raw = raw[raw["Equilibrium"].isin(EQ_levels)]

In [12]:
raw.shape

(301, 15)

In [13]:
raw.isna().sum()

Name             0
Type             0
Latitude         2
Longitude        2
Temperature      1
Depth          243
Cond            34
pH              79
TDS             37
Ca               1
Mg              10
Na               1
K                2
SiO2            11
Equilibrium      0
dtype: int64

In [14]:
raw['Equilibrium'].unique()

<StringArray>
['IM', 'FE', 'PE', 'PE/IM', 'FE/PE']
Length: 5, dtype: str

In [15]:
raw.dtypes

Name               str
Type               str
Latitude       float64
Longitude      float64
Temperature    float64
Depth           object
Cond           float64
pH             float64
TDS            float64
Ca             float64
Mg             float64
Na             float64
K              float64
SiO2           float64
Equilibrium        str
dtype: object

In [16]:
numeric_cols = ["Latitude", "Longitude", "Temperature", "Depth", "Cond", "pH",
                 "TDS", "Ca", "Mg", "Na", "K", "SiO2"]
for n in numeric_cols:
    raw[n] = pd.to_numeric(raw[n], errors="coerce")


In [17]:
raw.dtypes

Name               str
Type               str
Latitude       float64
Longitude      float64
Temperature    float64
Depth          float64
Cond           float64
pH             float64
TDS            float64
Ca             float64
Mg             float64
Na             float64
K              float64
SiO2           float64
Equilibrium        str
dtype: object

In [18]:
raw.isna().sum()

Name             0
Type             0
Latitude         2
Longitude        2
Temperature      1
Depth          251
Cond            34
pH              79
TDS             37
Ca               1
Mg              10
Na               1
K                2
SiO2            11
Equilibrium      0
dtype: int64

In [19]:
# Completeness filter
required_cols = ["Temperature", "Cond", "pH", "TDS", "Ca", "Mg", "Na", "K", "SiO2"]
mask = raw[required_cols].notna().all(axis=1)
data = raw[mask].copy().reset_index(drop=True)

In [20]:
data.isna().sum()

Name             0
Type             0
Latitude         0
Longitude        0
Temperature      0
Depth          161
Cond             0
pH               0
TDS              0
Ca               0
Mg               0
Na               0
K                0
SiO2             0
Equilibrium      0
dtype: int64

In [21]:
len(data)

196

In [22]:
data["Depth"] = data["Depth"].fillna(0.0)
# Binary target, IM = IM and FE, PE, FE/PE, PE/IM = EQ
EQ_codes = {"PE", "FE", "PE/IM", "FE/PE"}
data["EQ_Label"] = np.where(data["Equilibrium"].isin(EQ_codes), "EQ", "IM")
data["EQ_Label"].value_counts()

EQ_Label
IM    124
EQ     72
Name: count, dtype: int64

In [23]:
data["Type"].value_counts()

Type
Thermal Spring    153
Well               23
Artesian Well      20
Name: count, dtype: int64

In [24]:
out = ROOT / "data" / "processed"
out.mkdir(parents=True, exist_ok=True)
data.to_csv(out / "sample_196.csv", index=False)